# nb_30_gold_build

Gold- Builds the tables that people and reports actually use, from the clean silver tables.

The tables:
| Table | One row per | What it is |
|---|---|---|
| `gold_dim_client` | client record (one per region) | who the client is in that region, with consent flags |
| `gold_dim_product`, `gold_dim_boutique`, `gold_dim_date` | product / boutique / day | lookup tables |
| `gold_fact_sales` | sale or return line | amounts in EUR, linked to the client |
| `gold_fact_after_sales` | repair ticket | including repairs done in another region |
| `gold_fact_appointment` | appointment | |
| `gold_client_identity_map` | client record | which records belong to the same person, with the rule and confidence |
| `gold_client_360` | client record | one row of numbers per client **inside its own region** |
| `gold_person_360_global` | person | the combined view across regions, **only for people whose every record consents to sharing** |
| `gold_governance_summary` | metric | consent and sharing numbers for the report |

Two rules shape everything here
1. **Regions are separate by default-** `gold_client_360` only counts what happened in that client's own region.
2. **The combined profile is opt-in-** A person appears in `gold_person_360_global` only when every one of their records says yes to cross-region sharing. The other people are not in that table at all.

The tables are built here in the Lakehouse. The Warehouse scripts in the `sql/gold` folder copy them into the Warehouse and put the views and permissions on top.

Flags are stored as 0/1 numbers (not true/false) so the same SQL runs in Spark and in the Warehouse.

Input: all silver tables. Output: the `gold_*` tables above, plus rows in `ctl_dq_results` and `ctl_run_log`.


In [ ]:
# ---- Settings -------------------------------------------------------------------------
# The date everything is measured from. None = the latest sale date in the data.
AS_OF_DATE = None
LOOKBACK_DAYS = 365          # "trailing 12 months" for the tier

# Tier rules, on net sales in EUR over the lookback period. These are illustrative settings for the fictional house.
VIC_MIN_EUR = 350000
PRIORITY_MIN_EUR = 100000

In [ ]:
# ---- Setup ----------------------------------------------------------------------------
import uuid
from datetime import datetime, timezone, date
from functools import reduce

import pyspark.sql.functions as F
from pyspark.sql import Row, Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType,
                               DoubleType, TimestampType)

GOLD_BATCH_ID = "G" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
print("gold batch id:", GOLD_BATCH_ID)


def now_utc():
    return datetime.now(timezone.utc).replace(tzinfo=None)


dq_rows, run_rows, gaps = [], [], []


def add_dq(table, rule, severity, checked, failed):
    dq_rows.append(Row(batch_id=GOLD_BATCH_ID, layer="gold", table_name=table, rule=rule, severity=severity,
                       rows_checked=int(checked), rows_failed=int(failed),
                       pct_failed=round(100.0 * failed / checked, 3) if checked else 0.0, checked_at=now_utc()))


def write_gold(df, name):
    started = now_utc()
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    n = spark.table(name).count()
    run_rows.append(Row(batch_id=GOLD_BATCH_ID, layer="gold", object_name=name, status="success", files_loaded=0,
                        rows_loaded=int(n), started_at=started, finished_at=now_utc(), failure_class=None,
                        message=f"rows={n}"))
    print(f"{name:28s} {n:>8} rows")
    return n


def flag(cond):
    """True/false -> 1/0. A missing value counts as 0."""
    return F.when(cond, F.lit(1)).otherwise(F.lit(0))


def date_key(c):
    return F.date_format(c, "yyyyMMdd").cast("int")

In [ ]:
# ---- Lookup tables --------------------------------------------------------------------
sales = spark.table("silver_sales")
tickets = spark.table("silver_after_sales")
appts = spark.table("silver_appointment")

as_of = date.fromisoformat(AS_OF_DATE) if AS_OF_DATE else sales.agg(F.max("event_date")).collect()[0][0]
first_day = min(sales.agg(F.min("event_date")).collect()[0][0],
                tickets.agg(F.min(F.to_date("created_at"))).collect()[0][0],
                appts.agg(F.min(F.to_date("appointment_ts"))).collect()[0][0])
print("measured as of:", as_of, "| first day in data:", first_day)

n_product = write_gold(spark.table("silver_product"), "gold_dim_product")
n_boutique = write_gold(spark.table("silver_boutique"), "gold_dim_boutique")

days = spark.range(1).select(F.explode(F.sequence(F.lit(first_day), F.lit(as_of), F.expr("interval 1 day"))).alias("d"))
dim_date = days.select(date_key("d").alias("date_key"), F.col("d").alias("date"), F.year("d").alias("year"),
                       F.quarter("d").alias("quarter"), F.month("d").alias("month"), F.date_format("d", "MMM").alias("month_name"),
                       F.weekofyear("d").alias("week_of_year"), F.date_format("d", "E").alias("weekday"),
                       flag(F.dayofweek("d").isin(1, 7)).alias("is_weekend"))
n_date = write_gold(dim_date, "gold_dim_date")

In [ ]:
# ---- Clients and identity -------------------------------------------------------------
latest_w = Window.partitionBy("client_id").orderBy(F.col("updated_at").desc(), F.col("file_date").desc(), F.col("source_file").desc())
latest = (spark.table("silver_crm_client").withColumn("rk", F.row_number().over(latest_w)).filter("rk = 1"))
ident = spark.table("silver_client_identity")
person = spark.table("silver_person")

dim_client = (latest.join(ident.select("client_id", "person_key", "link_rule", "link_confidence"), "client_id")
              .join(person.select("person_key", "record_count", "consent_share_effective", "consent_conflict"), "person_key")
              .select("client_id", "region", "person_key", "first_name", "last_name", "home_boutique_id", "home_country",
                      F.col("created_at").cast("date").alias("created_date"),
                      flag(F.col("consent_share_cross_region")).alias("consent_share_cross_region"),
                      flag(F.col("consent_marketing")).alias("consent_marketing"),
                      flag(F.col("consent_share_effective")).alias("consent_share_effective"),
                      flag(F.col("consent_conflict")).alias("consent_conflict"),
                      flag(F.col("record_count") > 1).alias("is_multi_region_person"),
                      F.col("client_segment").alias("source_segment"), "link_rule", "link_confidence"))
n_client = write_gold(dim_client, "gold_dim_client")

id_map = (ident.join(dim_client.select("client_id", "is_multi_region_person", "consent_share_effective"), "client_id")
          .select("client_id", "person_key", "region", "cluster_size", "link_rule", "link_confidence",
                  flag(F.col("is_anchor")).alias("is_anchor"), "is_multi_region_person", "consent_share_effective"))
write_gold(id_map, "gold_client_identity_map")

In [ ]:
# ---- Facts ----------------------------------------------------------------------------
pk = spark.table("gold_dim_client").select("client_id", "person_key", F.col("region").alias("client_region"))

fact_sales = (sales.join(pk, "client_id", "left")
              .select("sale_id", date_key("event_date").alias("date_key"), "event_date", "client_id", "person_key",
                      F.col("client_region").alias("region"), "channel", "boutique_id", "sales_associate_id", "product_id", "category",
                      "quantity", "currency", "unit_price_eur", "line_amount_eur", flag(F.col("is_return")).alias("is_return"),
                      "original_sale_id", "days_to_return", flag(F.col("fx_carried_forward")).alias("fx_carried_forward")))
n_fs = write_gold(fact_sales, "gold_fact_sales")

fact_as = (tickets.join(pk, "client_id", "left")
           .select("ticket_id", date_key(F.to_date("created_at")).alias("created_date_key"), F.to_date("created_at").alias("created_date"),
                   F.to_date("closed_at").alias("closed_date"), "client_id", "person_key", F.col("client_region").alias("region"),
                   "boutique_id", "product_id", "status", "reason", "purchase_region", "service_region",
                   flag(F.col("is_cross_region_service")).alias("is_cross_region_service"), "days_to_close"))
n_fa = write_gold(fact_as, "gold_fact_after_sales")

fact_ap = (appts.join(pk, "client_id", "left")
           .select("appointment_id", date_key(F.to_date("appointment_ts")).alias("date_key"), F.to_date("appointment_ts").alias("appointment_date"),
                   "client_id", "person_key", F.col("client_region").alias("region"), "boutique_id", "advisor_id", "status"))
n_fp = write_gold(fact_ap, "gold_fact_appointment")

In [ ]:
# ---- Client 360, inside one region ----------------------------------------------------
fs = spark.table("gold_fact_sales")
since = F.date_sub(F.lit(as_of), LOOKBACK_DAYS)
sale_lines = F.col("is_return") == 0

per_client = fs.groupBy("client_id").agg(
    F.countDistinct(F.when(sale_lines, F.col("sale_id"))).alias("orders"),
    F.sum(F.when(F.col("is_return") == 1, 1).otherwise(0)).alias("return_lines"),
    F.round(F.sum("line_amount_eur"), 2).alias("net_sales_eur"),
    F.round(F.sum(F.when(sale_lines, F.col("line_amount_eur"))), 2).alias("gross_sales_eur"),
    F.round(F.sum(F.when(F.col("is_return") == 1, -F.col("line_amount_eur"))), 2).alias("returned_eur"),
    F.round(F.sum(F.when(F.col("event_date") > since, F.col("line_amount_eur"))), 2).alias("net_sales_12m_eur"),
    F.min(F.when(sale_lines, F.col("event_date"))).alias("first_purchase_date"),
    F.max(F.when(sale_lines, F.col("event_date"))).alias("last_purchase_date"),
    F.round(F.sum(F.when(sale_lines & (F.col("channel") == "boutique"), F.col("line_amount_eur"))) / F.sum(F.when(sale_lines, F.col("line_amount_eur"))), 3).alias("boutique_share"))

cat_w = Window.partitionBy("client_id").orderBy(F.col("spend").desc(), F.col("category"))
top_cat = (fs.filter(sale_lines).groupBy("client_id", "category").agg(F.sum("line_amount_eur").alias("spend"))
           .withColumn("rk", F.row_number().over(cat_w)).filter("rk = 1").select("client_id", F.col("category").alias("top_category")))

per_repair = spark.table("gold_fact_after_sales").groupBy("client_id").agg(
    F.count("*").alias("repairs"), F.sum(F.when(F.col("status") != "closed", 1).otherwise(0)).alias("open_repairs"),
    F.sum("is_cross_region_service").alias("cross_region_repairs"))
per_appt = spark.table("gold_fact_appointment").groupBy("client_id").agg(
    F.count("*").alias("appointments"), F.sum(F.when(F.col("status") == "completed", 1).otherwise(0)).alias("completed_appointments"),
    F.sum(F.when(F.col("status") == "no_show", 1).otherwise(0)).alias("no_shows"))

def tier(spend_col, orders_col):
    return (F.when(F.coalesce(spend_col, F.lit(0)) >= VIC_MIN_EUR, "vic")
             .when(F.coalesce(spend_col, F.lit(0)) >= PRIORITY_MIN_EUR, "priority")
             .when(F.coalesce(orders_col, F.lit(0)) > 0, "core").otherwise("prospect"))

zero = ["orders", "return_lines", "repairs", "open_repairs", "cross_region_repairs", "appointments", "completed_appointments", "no_shows"]
client_360 = (spark.table("gold_dim_client").select("client_id", "region", "consent_share_effective", "is_multi_region_person")
              .join(per_client, "client_id", "left").join(top_cat, "client_id", "left")
              .join(per_repair, "client_id", "left").join(per_appt, "client_id", "left")
              .fillna(0, subset=zero + ["net_sales_eur", "gross_sales_eur", "returned_eur", "net_sales_12m_eur"])
              .withColumn("days_since_last_purchase", F.datediff(F.lit(as_of), F.col("last_purchase_date")))
              .withColumn("tier", tier(F.col("net_sales_12m_eur"), F.col("orders")))
              .withColumn("as_of_date", F.lit(as_of).cast("date")))
n_360 = write_gold(client_360, "gold_client_360")

In [ ]:
# ---- Person 360, across regions: ONLY people whose every record consents --------------
# People without full consent are not in this table at all.
c360 = spark.table("gold_client_360")
c = spark.table("gold_dim_client").select("client_id", "person_key", "consent_share_effective")
shareable = c.filter("consent_share_effective = 1")

by_person = (shareable.join(c360.drop("consent_share_effective", "region", "is_multi_region_person"), "client_id")
             .groupBy("person_key").agg(
                 F.count("*").alias("record_count"),
                 F.sum("orders").alias("orders"), F.round(F.sum("net_sales_eur"), 2).alias("net_sales_eur"),
                 F.round(F.sum("gross_sales_eur"), 2).alias("gross_sales_eur"), F.round(F.sum("returned_eur"), 2).alias("returned_eur"),
                 F.round(F.sum("net_sales_12m_eur"), 2).alias("net_sales_12m_eur"),
                 F.min("first_purchase_date").alias("first_purchase_date"), F.max("last_purchase_date").alias("last_purchase_date"),
                 F.sum("repairs").alias("repairs"), F.sum("open_repairs").alias("open_repairs"),
                 F.sum("appointments").alias("appointments"), F.sum("no_shows").alias("no_shows")))
regions_by_person = (shareable.join(spark.table("gold_dim_client").select("client_id", "region"), "client_id")
                     .groupBy("person_key").agg(F.concat_ws(",", F.sort_array(F.collect_set("region"))).alias("regions"),
                                                F.size(F.collect_set("region")).alias("region_count")))
cat_p = (fs.join(c.select("client_id", "consent_share_effective"), "client_id").filter(sale_lines & (F.col("consent_share_effective") == 1))
         .groupBy("person_key", "category").agg(F.sum("line_amount_eur").alias("spend"))
         .withColumn("rk", F.row_number().over(Window.partitionBy("person_key").orderBy(F.col("spend").desc(), F.col("category"))))
         .filter("rk = 1").select("person_key", F.col("category").alias("top_category")))
person_global = (by_person.join(regions_by_person, "person_key").join(cat_p, "person_key", "left")
                 .withColumn("tier", tier(F.col("net_sales_12m_eur"), F.col("orders")))
                 .withColumn("days_since_last_purchase", F.datediff(F.lit(as_of), F.col("last_purchase_date")))
                 .withColumn("as_of_date", F.lit(as_of).cast("date")))
n_global = write_gold(person_global, "gold_person_360_global")

In [ ]:
# ---- Governance numbers for the report ------------------------------------------------
dc = spark.table("gold_dim_client")
persons = dc.groupBy("person_key").agg(F.max("is_multi_region_person").alias("multi"), F.max("consent_share_effective").alias("share"),
                                       F.max("consent_conflict").alias("conflict"))
fa = spark.table("gold_fact_after_sales").join(dc.select("client_id", "consent_share_effective"), "client_id", "left")
gov = [("people_total", persons.count()), ("people_in_several_regions", persons.filter("multi = 1").count()),
       ("people_in_several_regions_shareable", persons.filter("multi = 1 and share = 1").count()),
       ("people_in_several_regions_consent_conflict", persons.filter("multi = 1 and conflict = 1").count()),
       ("people_in_global_profile", n_global),
       ("repairs_total", fa.count()), ("repairs_in_another_region", fa.filter("is_cross_region_service = 1").count()),
       ("repairs_in_another_region_without_sharing_consent", fa.filter("is_cross_region_service = 1 and consent_share_effective = 0").count())]
write_gold(spark.createDataFrame([(m, int(v)) for m, v in gov], ["metric", "value"])
           .withColumn("as_of_date", F.lit(as_of).cast("date")), "gold_governance_summary")
spark.table("gold_governance_summary").show(truncate=False)

In [ ]:
# ---- Reconciliation: gold must agree with silver --------------------------------------
def total(df, col): return float(df.agg(F.sum(col)).collect()[0][0] or 0)

checks = [
    ("fact_sales rows = silver_sales rows", n_fs, sales.count()),
    ("fact_sales EUR total = silver_sales EUR total", round(total(spark.table("gold_fact_sales"), "line_amount_eur"), 2), round(total(sales, "line_amount_eur"), 2)),
    ("fact_after_sales rows = silver_after_sales rows", n_fa, tickets.count()),
    ("fact_appointment rows = silver_appointment rows", n_fp, appts.count()),
    ("client_360 rows = dim_client rows", n_360, n_client),
    ("client_360 EUR total = fact_sales EUR total", round(total(spark.table("gold_client_360"), "net_sales_eur"), 2), round(total(spark.table("gold_fact_sales"), "line_amount_eur"), 2)),
    ("sales rows whose client is not in dim_client", spark.table("gold_fact_sales").filter("person_key is null").count(), 0),
    ("tickets whose client is not in dim_client", spark.table("gold_fact_after_sales").filter("person_key is null").count(), 0),
    ("global profile: people without full consent", spark.table("gold_person_360_global").join(
        dc.groupBy("person_key").agg(F.min("consent_share_effective").alias("m")), "person_key").filter("m = 0").count(), 0),
    ("global profile EUR = sales of consenting people", round(total(spark.table("gold_person_360_global"), "net_sales_eur"), 2),
     round(total(fs.join(dc.select("client_id", "consent_share_effective"), "client_id").filter("consent_share_effective = 1"), "line_amount_eur"), 2)),
]
for name, got, want in checks:
    add_dq("gold", name, "reconcile", 1, 0 if got == want else 1)
    if got != want:
        gaps.append(f"{name} (got {got}, expected {want})")
spark.createDataFrame([(n, str(g), str(w), "ok" if g == w else "MISMATCH") for n, g, w in checks],
                      ["check", "gold", "expected", "result"]).show(truncate=False)

In [ ]:
# ---- Control tables -------------------------------------------------------------------
DQ_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("table_name", StringType()), StructField("rule", StringType()),
    StructField("severity", StringType()), StructField("rows_checked", LongType()),
    StructField("rows_failed", LongType()), StructField("pct_failed", DoubleType()),
    StructField("checked_at", TimestampType())])
RUN_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("object_name", StringType()), StructField("status", StringType()),
    StructField("files_loaded", IntegerType()), StructField("rows_loaded", LongType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()),
    StructField("failure_class", StringType()), StructField("message", StringType())])
spark.createDataFrame(dq_rows, schema=DQ_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_dq_results")
spark.createDataFrame(run_rows, schema=RUN_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_run_log")

if gaps:
    raise RuntimeError("Gold reconciliation failed for: " + "; ".join(gaps))
print("gold finished, batch", GOLD_BATCH_ID)

## Check the result
Read-only cells. Run them after the load.

In [ ]:
# Tiers: how many clients in each region, and what they bought in the last 12 months
(spark.table("gold_client_360").groupBy("region", "tier")
      .agg(F.count("*").alias("clients"), F.round(F.sum("net_sales_12m_eur") / 1e6, 2).alias("net_sales_12m_eur_millions"))
      .orderBy("region", "tier").show(20, truncate=False))

In [ ]:
# The combined profile: who is in it, and who is deliberately left out
dc = spark.table("gold_dim_client")
spark.createDataFrame([
    ("client records", dc.count()), ("people", dc.select("person_key").distinct().count()),
    ("people in the combined profile", spark.table("gold_person_360_global").count()),
    ("people left out (no full consent)", dc.groupBy("person_key").agg(F.min("consent_share_effective").alias("m")).filter("m = 0").count())],
    ["measure", "value"]).show(truncate=False)
spark.table("gold_person_360_global").groupBy("region_count", "tier").count().orderBy("region_count", "tier").show()

In [ ]:
# Repairs done in another region, and whether sharing consent existed
(spark.table("gold_fact_after_sales").join(spark.table("gold_dim_client").select("client_id", "consent_share_effective"), "client_id")
      .filter("is_cross_region_service = 1").groupBy("purchase_region", "service_region", "consent_share_effective").count()
      .orderBy("purchase_region", "service_region").show(truncate=False))

In [ ]:
# Look at one client in two regions: the regional rows and the combined row
p = (spark.table("gold_person_360_global").filter("region_count > 1").orderBy(F.col("net_sales_eur").desc()).select("person_key").first()["person_key"])
spark.table("gold_client_360").join(spark.table("gold_dim_client").filter(F.col("person_key") == p).select("client_id"), "client_id") \
     .select("client_id", "region", "orders", "net_sales_eur", "net_sales_12m_eur", "tier", "top_category").show(truncate=False)
spark.table("gold_person_360_global").filter(F.col("person_key") == p) \
     .select("regions", "orders", "net_sales_eur", "net_sales_12m_eur", "tier", "top_category").show(truncate=False)